<hr style="border: 6px solid#003262;" />

<div align="center">
    <img src="assets/content/images/nb_signals_correlation.png" align="center" width="20%">
</div>

<br>

# SIGNALS AND CORRELATION

<br>

**About:** A hands-on introduction to measuring and interpreting statistical correlation in Python using NumPy, Pandas, and SciPy.

**Learning Goals:** (1) Define Pearson correlation and derive it from covariance. (2) Compute correlation matrices with NumPy and Pandas. (3) Compare Pearson, Spearman, and Kendall-tau and choose the appropriate method. (4) Recognize when correlation does not imply causation.

**Keywords:** correlation, pearson, spearman, kendall, covariance, numpy, pandas, scipy, statistics

**Prerequisite Knowledge:** (1) Python basics, (2) NumPy arrays, (3) Pandas DataFrames, (4) Basic descriptive statistics (mean, standard deviation)

**Target User:** Self-learners seeking a practical foundation in correlation analysis for data science.

<hr style="border: 4px solid#003262;" />

<a name='Part_table_contents' id="Part_table_contents"></a>

#### CONTENTS

> #### [PART 0: SETUP](#Part_0)
> #### [PART 1: WHAT IS CORRELATION?](#Part_1)
> #### [PART 2: CORRELATION WITH NUMPY](#Part_2)
> #### [PART 3: CORRELATION WITH PANDAS](#Part_3)
> #### [PART 4: TYPES OF CORRELATION](#Part_4)

<br>

<a id='Part_0'></a>

<hr style="border: 2px solid#003262;" />

#### PART 0

## **SETUP**

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="assets/content/images/nb_signals_correlation.png" align="center" width="30%" padding="10"><br>
    <br>
</div>

All imports are collected here so every cell below runs after a clean kernel restart without any hidden state dependencies.

___

**Note:** This notebook uses NumPy for array-level correlation, Pandas for DataFrame-level correlation, Matplotlib for visualization, and SciPy for hypothesis-test-aware correlation functions. All four are standard in any data science Python environment.

___

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_1'></a>

<hr style="border: 2px solid#003262;" />

#### PART 1

## **WHAT** IS **CORRELATION?**

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="assets/content/images/nb_signals_correlation.png" align="center" width="30%" padding="10"><br>
    <br>
</div>

Correlation measures the **strength and direction** of the linear relationship between two variables. The result is a single number called the **correlation coefficient**, bounded between -1 and +1.

- A value near **+1** means the two variables tend to increase together.
- A value near **-1** means one tends to increase as the other decreases.
- A value near **0** means no consistent linear pattern exists between them.

The most common version is the **Pearson correlation coefficient**, defined as:

$$r = \frac{\sum_{i=1}^{n} (x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum_{i=1}^{n}(x_i - \bar{x})^2 \cdot \sum_{i=1}^{n}(y_i - \bar{y})^2}}$$

where:
- $x_i, y_i$ are the $i$-th observations of each variable,
- $\bar{x}, \bar{y}$ are the sample means,
- the numerator is the **covariance** of $x$ and $y$,
- the denominator normalizes by the product of each variable's standard deviation, which is why the result is always in $[-1, 1]$.

In short: Pearson correlation is standardized covariance. Covariance tells you whether two variables move together, but its scale depends on the units of the variables - dividing by the standard deviations removes that unit dependency.

___

<strong style="color:red">KEY CONSIDERATION:</strong> Correlation measures **linear** association only. Two variables can be strongly related (e.g., quadratically) and still have a near-zero Pearson correlation. Always visualize before concluding from a correlation number alone.

___

<strong style="color:red">KEY CONSIDERATION:</strong> Correlation does **not** imply causation. A high correlation between two variables may reflect a common cause, a coincidence in the data period, or a confounding third variable - not a direct causal link.

___

**Computing Pearson by Hand**

The cell below computes $r$ step-by-step using NumPy arithmetic - no library function calls. Walking through this once makes the formula concrete before we let `np.corrcoef` or `df.corr()` do the work.

In [ ]:
x = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
y = np.array([2.1, 3.9, 6.2, 7.8, 10.1])

x_mean = x.mean()
y_mean = y.mean()

numerator = np.sum((x - x_mean) * (y - y_mean))
denominator = np.sqrt(np.sum((x - x_mean)**2) * np.sum((y - y_mean)**2))

r = numerator / denominator
print(f"Pearson r (by hand): {r:.4f}")

# Verify using numpy's built-in
r_numpy = np.corrcoef(x, y)[0, 1]
print(f"Pearson r (np.corrcoef): {r_numpy:.4f}")

The two values should match exactly. `np.corrcoef` returns a 2x2 matrix - the off-diagonal element `[0, 1]` (or `[1, 0]`) is the correlation between the two input arrays.

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->


> **Given the two arrays below, compute the Pearson correlation coefficient by hand using NumPy arithmetic (not `np.corrcoef`). Then explain: what does the sign tell you? What does the magnitude tell you?**

<br>

In [ ]:
a = np.array([10.0, 20.0, 30.0, 40.0, 50.0])
b = np.array([50.0, 35.0, 25.0, 15.0, 5.0])

### YOUR CODE HERE ###
r_ab = ...

<hr style="border: 2px solid#003262;" />

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_2'></a>

<hr style="border: 2px solid#003262;" />

#### PART 2

## **CORRELATION** WITH **NUMPY**

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="assets/content/images/nb_signals_correlation.png" align="center" width="30%" padding="10"><br>
    <br>
</div>

NumPy's `np.corrcoef` computes the **correlation matrix** for a set of row vectors. Each entry $(i, j)$ in the result is the Pearson correlation between row $i$ and row $j$ of the input array. The diagonal is always 1 (every variable is perfectly correlated with itself).

The key detail is **what `np.corrcoef` treats as variables**: it treats each **row** as one variable. To correlate columns instead, transpose the input first.

In [ ]:
x = np.array([
    [0.1, .32, .2,  0.4, 0.8],
    [.23, .18, .56, .61, .12],
    [.9,  .3,  .6,  .5,  .3 ],
    [.34, .75, .91, .19, .21]
])

print("Shape:", x.shape)  # 4 rows, 5 columns
print()

# Correlation between rows: which rows move together?
corr_rows = np.corrcoef(x)
print("Row-to-row correlation matrix (4x4):")
print(np.round(corr_rows, 3))

**Reading the row correlation matrix:** entry `[0, 2]` is the correlation between row 0 and row 2. Because `np.corrcoef(x)` treats rows as variables, a 4-row input produces a 4x4 output.

Now transpose to correlate **columns** - which of the 5 columns (features) move together?

In [ ]:
# Correlation between columns: which features move together?
corr_cols = np.corrcoef(x.T)
print("Column-to-column correlation matrix (5x5):")
print(np.round(corr_cols, 3))

# Equivalent: np.corrcoef(x, rowvar=False) does the same thing
# rowvar=False tells NumPy to treat columns as variables instead of rows

___

**Note:** `np.corrcoef(x.T)` and `np.corrcoef(x, rowvar=False)` produce identical results. The `rowvar` parameter is explicit and may be clearer for readers unfamiliar with the transpose trick.

___

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->


> **Using the matrix `x` defined above, compute the column correlation matrix and identify which pair of columns has the weakest linear relationship (correlation closest to 0). Report both the column indices and the correlation value.**

<br>

In [ ]:
x = np.array([
    [0.1, .32, .2,  0.4, 0.8],
    [.23, .18, .56, .61, .12],
    [.9,  .3,  .6,  .5,  .3 ],
    [.34, .75, .91, .19, .21]
])

### YOUR CODE HERE ###
corr_cols = ...

# Find the weakest pair (exclude diagonal)
...

<hr style="border: 2px solid#003262;" />

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_3'></a>

<hr style="border: 2px solid#003262;" />

#### PART 3

## **CORRELATION** WITH **PANDAS**

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="assets/content/images/nb_signals_correlation.png" align="center" width="30%" padding="10"><br>
    <br>
</div>

Pandas makes it easy to compute pairwise correlations across all columns of a DataFrame in one call. This is especially useful during exploratory data analysis, where you want to quickly identify which features move together before building a model.

The dataset below is a subset of the classic `mtcars` dataset - six measurements on six cars. The columns represent miles-per-gallon (mpg), engine displacement (disp), horsepower (hp), rear axle ratio (drat), weight (wt), and quarter-mile time (qsec).

In [ ]:
d = {
    'mpg':  [21.0, 21.0, 22.8, 21.4, 18.7, 18.1],
    'disp': [160,  160,  108,  258,  360,  225 ],
    'hp':   [110,  110,  93,   110,  175,  105 ],
    'drat': [3.90, 3.90, 3.85, 3.08, 3.15, 2.76],
    'wt':   [2.620, 2.875, 2.320, 3.215, 3.440, 3.460],
    'qsec': [16.46, 17.02, 18.61, 19.44, 17.02, 20.22]
}
index = ['Mazda RX4', 'Mazda RX4 Wag', 'Datsun 710',
         'Hornet 4 Drive', 'Hornet Sportabout', 'Valiant']

df = pd.DataFrame(data=d, index=index)
df

**Full pairwise correlation matrix.** `df.corr()` computes the Pearson correlation between every pair of numeric columns by default. The result is a square symmetric matrix - reading the row for `mpg` shows how miles-per-gallon relates to every other feature.

In [ ]:
# Default method='pearson' - can also pass 'spearman' or 'kendall'
corr_matrix = df.corr()
print(corr_matrix.round(3))

**Single-pair correlation.** When you only need one value, use the column `.corr()` method directly. This is more readable than indexing into a full matrix.

In [ ]:
# Pearson correlation between mpg and hp
r_mpg_hp = df['mpg'].corr(df['hp'])
print(f"Pearson r(mpg, hp) = {r_mpg_hp:.4f}")

# Switch method in-place - no need to rebuild the DataFrame
r_spearman = df['mpg'].corr(df['hp'], method='spearman')
print(f"Spearman r(mpg, hp) = {r_spearman:.4f}")

___

**Note:** `df.corr()` computes pairwise complete cases by default - if two columns both have a non-NaN value for a given row, that row is included in their pairwise correlation, even if other columns are NaN. This means the effective sample size can differ across pairs in a DataFrame with missing values.

___

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->


> **Using the `df` DataFrame above, compute the **Spearman** correlation between `mpg` and `wt`. Then compute the **Pearson** correlation between the same pair. Are the two values similar? What would a large difference between them suggest about the data?**

<br>

In [ ]:
### YOUR CODE HERE ###
r_pearson = ...
r_spearman = ...

print(f"Pearson:  {r_pearson:.4f}")
print(f"Spearman: {r_spearman:.4f}")

<hr style="border: 2px solid#003262;" />

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_4'></a>

<hr style="border: 2px solid#003262;" />

#### PART 4

## **TYPES** OF **CORRELATION**

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="assets/content/images/nb_signals_correlation.png" align="center" width="30%" padding="10"><br>
    <br>
</div>

Three correlation methods appear frequently in data science. They answer related but distinct questions:

| Method | Measures | Assumes | Sensitive to outliers? | Best when |
|---|---|---|---|---|
| **Pearson** | Linear relationship | Continuous data, normality helpful | Yes | Both variables are continuous and roughly normal |
| **Spearman** | Monotonic relationship (rank-based) | Ordinal or continuous data | No | Data has outliers, is skewed, or is ordinal |
| **Kendall-tau** | Concordant vs. discordant pairs | Ordinal or continuous data | No | Small samples, many tied ranks, or when a p-value with fewer assumptions is needed |

**Pearson** standardizes covariance. It captures whether two variables move together linearly - equal steps in one variable correspond to equal steps in the other.

**Spearman** converts each variable to ranks, then applies Pearson to the ranks. Because ranks compress outlier influence, Spearman is more robust when data is skewed or contains extreme values. A Spearman coefficient of +1 only requires a monotonically increasing relationship - not a linear one.

**Kendall-tau** counts pairs of observations: a pair $(x_i, y_i), (x_j, y_j)$ is **concordant** if the ordering of $x$ matches the ordering of $y$ (both $x_i > x_j$ and $y_i > y_j$, or both less), and **discordant** if the orderings disagree. The coefficient is the proportion of concordant pairs minus discordant pairs, normalized to $[-1, 1]$. Kendall-tau is preferred over Spearman on small samples with many ties because its sampling distribution is better understood in those conditions.

In [ ]:
# All three methods on the same pair of arrays
x = np.array([1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0])
y = np.array([1.5, 3.8, 2.9, 5.1, 6.0, 5.8, 8.3, 9.2])

# TODO: verify against current scipy docs - function names stable since 1.x but check parameters
r_p, p_p   = stats.pearsonr(x, y)
r_s, p_s   = stats.spearmanr(x, y)
r_k, p_k   = stats.kendalltau(x, y)

print(f"Pearson  r = {r_p:.4f}  (p = {p_p:.4f})")
print(f"Spearman r = {r_s:.4f}  (p = {p_s:.4f})")
print(f"Kendall  t = {r_k:.4f}  (p = {p_k:.4f})")

**Effect of outliers.** The main practical difference between Pearson and the rank-based methods becomes clear when the data has one or two extreme values. The cell below injects a single outlier and shows how each method responds.

In [ ]:
# Clean data
x_clean = np.array([1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0])
y_clean = np.array([1.5, 3.8, 2.9, 5.1, 6.0, 5.8, 8.3, 9.2])

# Data with one extreme outlier added
x_out = np.append(x_clean, 100.0)
y_out = np.append(y_clean, 0.5)

for label, xv, yv in [("Clean  ", x_clean, y_clean), ("Outlier", x_out, y_out)]:
    rp, _ = stats.pearsonr(xv, yv)
    rs, _ = stats.spearmanr(xv, yv)
    rk, _ = stats.kendalltau(xv, yv)
    print(f"{label} | Pearson {rp:+.3f} | Spearman {rs:+.3f} | Kendall {rk:+.3f}")

The Pearson coefficient shifts substantially with the outlier because it operates on the raw values. Spearman and Kendall shift far less because ranks compress extreme values.

___

**Note:** SciPy's correlation functions return a two-element tuple: the coefficient and the two-tailed p-value. The p-value tests the null hypothesis that the correlation is zero in the population - a small p-value suggests the observed correlation is unlikely under that null. Use the p-value as one signal among many, not as a binary gate.

___

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->


> **You are analyzing the relationship between household income (heavily right-skewed, with a few very high earners) and years of education in a small survey of 30 respondents. Which correlation method would you choose and why? Compute all three on the arrays below to support your reasoning.**

<br>

In [ ]:
income    = np.array([28000, 35000, 42000, 38000, 55000, 61000, 72000,
                        48000, 39000, 95000, 105000, 51000, 44000, 67000,
                        250000, 33000, 46000, 58000, 71000, 43000])
education = np.array([12, 14, 16, 14, 18, 18, 20, 16, 14, 22, 20, 16,
                      16, 18, 22, 13, 16, 18, 20, 15])

### YOUR CODE HERE ###
r_pearson  = ...
r_spearman = ...
r_kendall  = ...

<hr style="border: 2px solid#003262;" />

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<hr style="border: 6px solid#003262;" />